In [1]:
import subprocess, sys, socket, time

subprocess.run([sys.executable, '-m', 'pip', 'install', 'lightgbm', '-q'], check=True)
print('lightgbm OK')

def port_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(1)
        return s.connect_ex(('127.0.0.1', port)) == 0

if not port_open(6000):
    print('MLflow no responde — iniciando...')
    proc = subprocess.Popen(['bash', '/srv/start_mlflow.sh'])
    print('MLflow PID:', proc.pid)
    time.sleep(5)
    if port_open(6000):
        print('MLflow OK — http://localhost:6000')
    else:
        print('[ERROR] MLflow no levanto — verificar /srv/start_mlflow.sh')
else:
    print('MLflow ya estaba corriendo — http://localhost:6000')

import mlflow
MLFLOW_EXPERIMENT = 'latam2cloud'
mlflow.set_tracking_uri('http://localhost:6000')
mlflow.set_experiment(MLFLOW_EXPERIMENT)
print('MLflow OK — experimento:', MLFLOW_EXPERIMENT)



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python3.11 -m pip install --upgrade pip


lightgbm OK
MLflow no responde — iniciando...
MLflow PID: 4737
MLflow PID: 4738
MLflow OK — http://localhost:6000
MLflow OK — experimento: latam2cloud


In [2]:
import numpy as np
import pandas as pd
import warnings
import gc
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import cohen_kappa_score, f1_score, balanced_accuracy_score, recall_score
import lightgbm as lgb
from datetime import datetime

warnings.filterwarnings('ignore')

IN_FEAT    = '/srv/work/te_features_v6.parquet'
ROUTE_KEYS = ['source_agent', 'cloud_provider', 'region']
TARGET_COL = 'label_ventana'
HORIZON    = 2

RF_PARAMS = dict(
    n_estimators=300,
    max_features='sqrt',
    min_samples_leaf=5,
    class_weight='balanced_subsample',
    max_samples=500_000,
    n_jobs=2,
    random_state=42,
)

LGB_PARAMS = dict(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight='balanced',
    random_state=42,
    n_jobs=4,
    verbose=-1,
)

print('Imports OK —', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('RF params  :', RF_PARAMS)
print('LGB params :', LGB_PARAMS)


Imports OK — 2026-10-12 04:40:57
RF params  : {'n_estimators': 300, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'class_weight': 'balanced_subsample', 'max_samples': 500000, 'n_jobs': 2, 'random_state': 42}
LGB params : {'n_estimators': 500, 'learning_rate': 0.05, 'num_leaves': 63, 'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8, 'class_weight': 'balanced', 'random_state': 42, 'n_jobs': 4, 'verbose': -1}


In [3]:
import gc
import numpy as np
import os
import pyarrow as pa
import pyarrow.parquet as pq

TMP_DIR = '/srv/work/ablacion_tmp'
os.makedirs(TMP_DIR, exist_ok=True)

pf = pq.ParquetFile(IN_FEAT)
schema = pf.schema_arrow
schema_names = {f.name for f in schema}
META_COLS_SET = set(ROUTE_KEYS) | {TARGET_COL, 'split', 'timestamp', 'window_start',
                                     'window_end', 'fecha', 'window_id', 'ts', 'date'}
feat_cols  = [
    f.name for f in schema
    if f.name not in META_COLS_SET
    and (pa.types.is_floating(f.type) or pa.types.is_integer(f.type))
]
bgp_cols   = [c for c in feat_cols if c.startswith('bgp_')]
nobgp_cols = [c for c in feat_cols if not c.startswith('bgp_')]
FEAT_A = nobgp_cols
FEAT_B = feat_cols
FEAT_C = bgp_cols
print('feat_cols:', len(feat_cols), ' BGP:', len(bgp_cols))

needed_cols = [c for c in ROUTE_KEYS + [TARGET_COL, 'split'] if c in schema_names]
meta_df = pq.read_table(IN_FEAT, columns=needed_cols).to_pandas()
print('meta_df: %.0f MB' % (meta_df.memory_usage(deep=True).sum() / 1024**2))

meta_df['y'] = meta_df.groupby(ROUTE_KEYS)[TARGET_COL].shift(-HORIZON)
valid_mask  = meta_df['y'].notna()
mask_train  = (meta_df['split'] == 'train') & valid_mask
mask_val    = (meta_df['split'] == 'val')   & valid_mask

y_train = meta_df.loc[mask_train, 'y'].values.astype(np.int32).copy()
y_val   = meta_df.loc[mask_val,   'y'].values.astype(np.int32).copy()
n_train = int(mask_train.sum())
n_val   = int(mask_val.sum())
print('Train:', n_train, ' Val:', n_val)

np.save(os.path.join(TMP_DIR, 'mask_train.npy'), mask_train.values)
np.save(os.path.join(TMP_DIR, 'mask_val.npy'),   mask_val.values)
np.save(os.path.join(TMP_DIR, 'y_train.npy'),    y_train)
np.save(os.path.join(TMP_DIR, 'y_val.npy'),      y_val)

del meta_df, mask_train, mask_val, valid_mask
gc.collect()
print('Masks guardadas — RAM libre para Cell 3')

print('Feature sets:')
print('  Config A (sin BGP)  :', len(FEAT_A), 'features')
print('  Config B (con BGP)  :', len(FEAT_B), 'features')
print('  Config C (solo BGP) :', len(FEAT_C), 'features')


feat_cols: 103  BGP: 3
meta_df: 612 MB
Train: 5349594  Val: 1140847
Masks guardadas — RAM libre para Cell 3
Feature sets:
  Config A (sin BGP)  : 100 features
  Config B (con BGP)  : 103 features
  Config C (solo BGP) : 3 features


In [4]:
import gc
import numpy as np
import os
import pyarrow as pa
import pyarrow.parquet as pq

TMP_DIR = '/srv/work/ablacion_tmp'

mask_train = np.load(os.path.join(TMP_DIR, 'mask_train.npy'))
mask_val   = np.load(os.path.join(TMP_DIR, 'mask_val.npy'))
y_train    = np.load(os.path.join(TMP_DIR, 'y_train.npy'))
y_val      = np.load(os.path.join(TMP_DIR, 'y_val.npy'))

n_train = int(mask_train.sum())
n_val   = int(mask_val.sum())
print('Train:', n_train, ' Val:', n_val)
print('Clase train:', dict(zip(*np.unique(y_train, return_counts=True))))

configs = [
    ('tr_A', mask_train, FEAT_A),
    ('va_A', mask_val,   FEAT_A),
    ('tr_B', mask_train, FEAT_B),
    ('va_B', mask_val,   FEAT_B),
    ('tr_C', mask_train, FEAT_C),
    ('va_C', mask_val,   FEAT_C),
]

memmaps = {}
for tag, mask, features in configs:
    n_rows = int(mask.sum())
    path = os.path.join(TMP_DIR, 'X_' + tag + '.npy')
    memmaps[tag] = np.lib.format.open_memmap(
        path, mode='w+', dtype='float32', shape=(n_rows, len(features)))

idx_A = {col: j for j, col in enumerate(FEAT_A)}
idx_B = {col: j for j, col in enumerate(FEAT_B)}
idx_C = {col: j for j, col in enumerate(FEAT_C)}

BATCH = 10
total_batches = (len(feat_cols) + BATCH - 1) // BATCH

for bi in range(total_batches):
    batch_cols = feat_cols[bi * BATCH : (bi + 1) * BATCH]
    tbl = pq.read_table(IN_FEAT, columns=batch_cols)

    for col_name in batch_cols:
        arr = tbl.column(col_name)
        if arr.type == pa.float32():
            pass
        elif pa.types.is_floating(arr.type) or pa.types.is_integer(arr.type):
            arr = arr.cast(pa.float32())
        else:
            del arr
            continue
        col_np = arr.to_pandas().to_numpy()
        del arr

        nan_mask = np.isnan(col_np)
        if nan_mask.any():
            fill = np.float32(1.0) if 'ratio' in col_name.lower() else np.float32(0.0)
            col_np = col_np.copy()
            col_np[nan_mask] = fill

        if col_name in idx_A:
            memmaps['tr_A'][:, idx_A[col_name]] = col_np[mask_train]
            memmaps['va_A'][:, idx_A[col_name]] = col_np[mask_val]
        if col_name in idx_B:
            memmaps['tr_B'][:, idx_B[col_name]] = col_np[mask_train]
            memmaps['va_B'][:, idx_B[col_name]] = col_np[mask_val]
        if col_name in idx_C:
            memmaps['tr_C'][:, idx_C[col_name]] = col_np[mask_train]
            memmaps['va_C'][:, idx_C[col_name]] = col_np[mask_val]
        del col_np

    del tbl
    gc.collect()
    print('Batch %d/%d OK' % (bi + 1, total_batches))

for tag in list(memmaps.keys()):
    del memmaps[tag]
memmaps = {}
gc.collect()

for tag, mask, features in configs:
    mb = int(mask.sum()) * len(features) * 4 // 1024**2
    print('X_' + tag + '.npy — %d MB' % mb)

print('Matrices guardadas — RAM lista para entrenamiento')


Train: 5349594  Val: 1140847
Clase train: {0: 4621399, 1: 509392, 2: 112220, 3: 106583}
Batch 1/11 OK
Batch 2/11 OK
Batch 3/11 OK
Batch 4/11 OK
Batch 5/11 OK
Batch 6/11 OK
Batch 7/11 OK
Batch 8/11 OK
Batch 9/11 OK
Batch 10/11 OK
Batch 11/11 OK
X_tr_A.npy — 2040 MB
X_va_A.npy — 435 MB
X_tr_B.npy — 2101 MB
X_va_B.npy — 448 MB
X_tr_C.npy — 61 MB
X_va_C.npy — 13 MB
Matrices guardadas — RAM lista para entrenamiento


In [5]:
def eval_model(y_true, y_pred, label=''):
    qwk        = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    rec_alta   = recall_score(y_true, y_pred, labels=[2], average='macro', zero_division=0)
    macro_f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)
    bal_acc    = balanced_accuracy_score(y_true, y_pred)
    if label:
        print(f'  {label}')
        print(f'    QWK={qwk:.4f}  RecAlta={rec_alta:.4f}  MacroF1={macro_f1:.4f}  BalAcc={bal_acc:.4f}')
    return {'qwk': qwk, 'recall_alta': rec_alta, 'macro_f1': macro_f1, 'bal_acc': bal_acc}

resultados = {}
print('eval_model() definida.')

eval_model() definida.


In [6]:
import os
TMP_DIR = '/srv/work/ablacion_tmp'
t0 = datetime.now()
X_tr_A = np.load(os.path.join(TMP_DIR, 'X_tr_A.npy'))
X_va_A = np.load(os.path.join(TMP_DIR, 'X_va_A.npy'))
print('RF Config A — sin BGP (%d features) — inicio %s' % (X_tr_A.shape[1], t0.strftime('%H:%M:%S')))

rf_A = RandomForestClassifier(**RF_PARAMS)
rf_A.fit(X_tr_A, y_train)
y_pred_rf_A = rf_A.predict(X_va_A)
resultados['rf_A'] = eval_model(y_val, y_pred_rf_A, 'RF Config A (sin BGP) — Validacion')

with mlflow.start_run(run_name='rf_A_sinBGP'):
    mlflow.log_params(RF_PARAMS)
    mlflow.log_param('config', 'A')
    mlflow.log_param('n_features', X_tr_A.shape[1])
    mlflow.log_param('dataset', 'v6_162d')
    mlflow.log_metrics(resultados['rf_A'])

elapsed = (datetime.now() - t0).seconds
print('RF Config A listo en %ds.' % elapsed)
del X_tr_A, X_va_A, rf_A; gc.collect()


RF Config A — sin BGP (100 features) — inicio 14:36:49
  RF Config A (sin BGP) — Validacion
    QWK=0.6260  RecAlta=0.3882  MacroF1=0.6508  BalAcc=0.6560
🏃 View run rf_A_sinBGP at: http://localhost:6000/#/experiments/681062908553326840/runs/4d9249675db94ab4a1fd6fcbd8fdd323
🧪 View experiment at: http://localhost:6000/#/experiments/681062908553326840
RF Config A listo en 1829s.


462

In [7]:
import os
TMP_DIR = '/srv/work/ablacion_tmp'
t0 = datetime.now()
X_tr_B = np.load(os.path.join(TMP_DIR, 'X_tr_B.npy'))
X_va_B = np.load(os.path.join(TMP_DIR, 'X_va_B.npy'))
print('RF Config B — con BGP (%d features) — inicio %s' % (X_tr_B.shape[1], t0.strftime('%H:%M:%S')))

rf_B = RandomForestClassifier(**RF_PARAMS)
rf_B.fit(X_tr_B, y_train)
y_pred_rf_B = rf_B.predict(X_va_B)
resultados['rf_B'] = eval_model(y_val, y_pred_rf_B, 'RF Config B (con BGP, D37) — Validacion')

with mlflow.start_run(run_name='rf_B_conBGP'):
    mlflow.log_params(RF_PARAMS)
    mlflow.log_param('config', 'B')
    mlflow.log_param('n_features', X_tr_B.shape[1])
    mlflow.log_param('dataset', 'v6_162d')
    mlflow.log_metrics(resultados['rf_B'])

elapsed = (datetime.now() - t0).seconds
print('RF Config B listo en %ds.' % elapsed)
del X_tr_B, X_va_B, rf_B; gc.collect()


RF Config B — con BGP (103 features) — inicio 15:30:30
  RF Config B (con BGP, D37) — Validacion
    QWK=0.6260  RecAlta=0.3828  MacroF1=0.6497  BalAcc=0.6557
🏃 View run rf_B_conBGP at: http://localhost:6000/#/experiments/681062908553326840/runs/fe138371b7224ccf9f8e3e6df1a493d3
🧪 View experiment at: http://localhost:6000/#/experiments/681062908553326840
RF Config B listo en 1834s.


90

In [8]:
import os
TMP_DIR = '/srv/work/ablacion_tmp'
t0 = datetime.now()
X_tr_C = np.load(os.path.join(TMP_DIR, 'X_tr_C.npy'))
X_va_C = np.load(os.path.join(TMP_DIR, 'X_va_C.npy'))
print('RF Config C — solo BGP (%d features) — inicio %s' % (X_tr_C.shape[1], t0.strftime('%H:%M:%S')))

rf_C = RandomForestClassifier(**RF_PARAMS)
rf_C.fit(X_tr_C, y_train)
y_pred_rf_C = rf_C.predict(X_va_C)
resultados['rf_C'] = eval_model(y_val, y_pred_rf_C, 'RF Config C (solo BGP) — Validacion')

with mlflow.start_run(run_name='rf_C_soloBGP'):
    mlflow.log_params(RF_PARAMS)
    mlflow.log_param('config', 'C')
    mlflow.log_param('n_features', X_tr_C.shape[1])
    mlflow.log_param('dataset', 'v6_162d')
    mlflow.log_metrics(resultados['rf_C'])

elapsed = (datetime.now() - t0).seconds
print('RF Config C listo en %ds.' % elapsed)
del X_tr_C, X_va_C, rf_C; gc.collect()


RF Config C — solo BGP (3 features) — inicio 16:17:36
  RF Config C (solo BGP) — Validacion
    QWK=0.0131  RecAlta=0.0556  MacroF1=0.2374  BalAcc=0.2588
🏃 View run rf_C_soloBGP at: http://localhost:6000/#/experiments/681062908553326840/runs/a149eef411a94388989a361ed7c49b9c
🧪 View experiment at: http://localhost:6000/#/experiments/681062908553326840
RF Config C listo en 435s.


576

In [9]:
import os
TMP_DIR = '/srv/work/ablacion_tmp'
t0 = datetime.now()
X_tr_A = np.load(os.path.join(TMP_DIR, 'X_tr_A.npy'))
X_va_A = np.load(os.path.join(TMP_DIR, 'X_va_A.npy'))
print('LGB Config A — sin BGP (%d features) — inicio %s' % (X_tr_A.shape[1], t0.strftime('%H:%M:%S')))

lgb_A = lgb.LGBMClassifier(**LGB_PARAMS)
lgb_A.fit(X_tr_A, y_train)
y_pred_lgb_A = lgb_A.predict(X_va_A)
resultados['lgb_A'] = eval_model(y_val, y_pred_lgb_A, 'LGB Config A (sin BGP) — Validacion')

with mlflow.start_run(run_name='lgb_A_sinBGP'):
    mlflow.log_params(LGB_PARAMS)
    mlflow.log_param('config', 'A')
    mlflow.log_param('n_features', X_tr_A.shape[1])
    mlflow.log_param('dataset', 'v6_162d')
    mlflow.log_metrics(resultados['lgb_A'])

elapsed = (datetime.now() - t0).seconds
print('LGB Config A listo en %ds.' % elapsed)
del X_tr_A, X_va_A, lgb_A; gc.collect()


LGB Config A — sin BGP (100 features) — inicio 16:31:10
  LGB Config A (sin BGP) — Validacion
    QWK=0.4493  RecAlta=0.6599  MacroF1=0.5710  BalAcc=0.7554
🏃 View run lgb_A_sinBGP at: http://localhost:6000/#/experiments/681062908553326840/runs/dad2fae7a87c426899bfd83f592aaabf
🧪 View experiment at: http://localhost:6000/#/experiments/681062908553326840
LGB Config A listo en 786s.


94

In [10]:
import os
TMP_DIR = '/srv/work/ablacion_tmp'
t0 = datetime.now()
X_tr_B = np.load(os.path.join(TMP_DIR, 'X_tr_B.npy'))
X_va_B = np.load(os.path.join(TMP_DIR, 'X_va_B.npy'))
print('LGB Config B — con BGP (%d features) — inicio %s' % (X_tr_B.shape[1], t0.strftime('%H:%M:%S')))

lgb_B = lgb.LGBMClassifier(**LGB_PARAMS)
lgb_B.fit(X_tr_B, y_train)
y_pred_lgb_B = lgb_B.predict(X_va_B)
resultados['lgb_B'] = eval_model(y_val, y_pred_lgb_B, 'LGB Config B (con BGP) — Validacion')

with mlflow.start_run(run_name='lgb_B_conBGP'):
    mlflow.log_params(LGB_PARAMS)
    mlflow.log_param('config', 'B')
    mlflow.log_param('n_features', X_tr_B.shape[1])
    mlflow.log_param('dataset', 'v6_162d')
    mlflow.log_metrics(resultados['lgb_B'])

elapsed = (datetime.now() - t0).seconds
print('LGB Config B listo en %ds.' % elapsed)
del X_tr_B, X_va_B, lgb_B; gc.collect()


LGB Config B — con BGP (103 features) — inicio 16:44:36
  LGB Config B (con BGP) — Validacion
    QWK=0.4380  RecAlta=0.6612  MacroF1=0.5656  BalAcc=0.7578
🏃 View run lgb_B_conBGP at: http://localhost:6000/#/experiments/681062908553326840/runs/70132c304e274cfeb6196b1c550fa5fa
🧪 View experiment at: http://localhost:6000/#/experiments/681062908553326840
LGB Config B listo en 799s.


0

In [11]:
import os
TMP_DIR = '/srv/work/ablacion_tmp'
t0 = datetime.now()
X_tr_C = np.load(os.path.join(TMP_DIR, 'X_tr_C.npy'))
X_va_C = np.load(os.path.join(TMP_DIR, 'X_va_C.npy'))
print('LGB Config C — solo BGP (%d features) — inicio %s' % (X_tr_C.shape[1], t0.strftime('%H:%M:%S')))

lgb_C = lgb.LGBMClassifier(**LGB_PARAMS)
lgb_C.fit(X_tr_C, y_train)
y_pred_lgb_C = lgb_C.predict(X_va_C)
resultados['lgb_C'] = eval_model(y_val, y_pred_lgb_C, 'LGB Config C (solo BGP) — Validacion')

with mlflow.start_run(run_name='lgb_C_soloBGP'):
    mlflow.log_params(LGB_PARAMS)
    mlflow.log_param('config', 'C')
    mlflow.log_param('n_features', X_tr_C.shape[1])
    mlflow.log_param('dataset', 'v6_162d')
    mlflow.log_metrics(resultados['lgb_C'])

elapsed = (datetime.now() - t0).seconds
print('LGB Config C listo en %ds.' % elapsed)
del X_tr_C, X_va_C, lgb_C; gc.collect()


LGB Config C — solo BGP (3 features) — inicio 16:58:04
  LGB Config C (solo BGP) — Validacion
    QWK=0.0040  RecAlta=0.2793  MacroF1=0.1093  BalAcc=0.2512
🏃 View run lgb_C_soloBGP at: http://localhost:6000/#/experiments/681062908553326840/runs/744fe504b9b3447693a82132c8427cea
🧪 View experiment at: http://localhost:6000/#/experiments/681062908553326840
LGB Config C listo en 245s.


528

In [12]:
def delta(b, a, key):
    if key in b and key in a:
        return b[key] - a[key]
    return float('nan')

rows = [
    ('RF',  'A (sin BGP)',  len(FEAT_A), resultados.get('rf_A',  {})),
    ('RF',  'B (con BGP)',  len(FEAT_B), resultados.get('rf_B',  {})),
    ('RF',  'C (solo BGP)', len(FEAT_C), resultados.get('rf_C',  {})),
    ('LGB', 'A (sin BGP)',  len(FEAT_A), resultados.get('lgb_A', {})),
    ('LGB', 'B (con BGP)',  len(FEAT_B), resultados.get('lgb_B', {})),
    ('LGB', 'C (solo BGP)', len(FEAT_C), resultados.get('lgb_C', {})),
]

print(f'{"Modelo":<6} {"Config":<14} {"N feat":>6}  {"QWK":>6}  {"RecAlta":>7}  {"MacroF1":>7}  {"BalAcc":>7}')
print('-' * 68)
for mdl, cfg, nf, r in rows:
    qwk = r.get('qwk', float('nan'))
    rec = r.get('recall_alta', float('nan'))
    f1  = r.get('macro_f1', float('nan'))
    ba  = r.get('bal_acc', float('nan'))
    print(f'{mdl:<6} {cfg:<14} {nf:>6}  {qwk:>6.4f}  {rec:>7.4f}  {f1:>7.4f}  {ba:>7.4f}')

print('\n── ΔB-A (contribución BGP) ──')
for mdl, key_a, key_b in [('RF', 'rf_A', 'rf_B'), ('LGB', 'lgb_A', 'lgb_B')]:
    dqwk = delta(resultados.get(key_b, {}), resultados.get(key_a, {}), 'qwk')
    drec = delta(resultados.get(key_b, {}), resultados.get(key_a, {}), 'recall_alta')
    sign = '+' if dqwk > 0 else ''
    print(f'  {mdl}  ΔQWK={sign}{dqwk:.4f}   ΔRecAlta={sign}{drec:.4f}')

print('\n── H1 indicador: ΔB-A ≤ 0 en ambos modelos → H1 VERIFICADA ──')

Modelo Config         N feat     QWK  RecAlta  MacroF1   BalAcc
--------------------------------------------------------------------
RF     A (sin BGP)       100  0.6260   0.3882   0.6508   0.6560
RF     B (con BGP)       103  0.6260   0.3828   0.6497   0.6557
RF     C (solo BGP)        3  0.0131   0.0556   0.2374   0.2588
LGB    A (sin BGP)       100  0.4493   0.6599   0.5710   0.7554
LGB    B (con BGP)       103  0.4380   0.6612   0.5656   0.7578
LGB    C (solo BGP)        3  0.0040   0.2793   0.1093   0.2512

── ΔB-A (contribución BGP) ──
  RF  ΔQWK=-0.0000   ΔRecAlta=-0.0055
  LGB  ΔQWK=-0.0113   ΔRecAlta=0.0013

── H1 indicador: ΔB-A ≤ 0 en ambos modelos → H1 VERIFICADA ──


In [13]:
import os
TMP_DIR = '/srv/work/ablacion_tmp'
X_tr_B = np.load(os.path.join(TMP_DIR, 'X_tr_B.npy'))
X_va_B = np.load(os.path.join(TMP_DIR, 'X_va_B.npy'))
rf_B2 = RandomForestClassifier(**RF_PARAMS)
rf_B2.fit(X_tr_B, y_train)

imp_B = pd.Series(rf_B2.feature_importances_, index=FEAT_B).sort_values(ascending=False)
bgp_imp   = imp_B[bgp_cols].sum()
total_imp = imp_B.sum()
bgp_pct   = bgp_imp / total_imp * 100

print('RF — Importancia BGP total (MDI): %.4f  (%.2f%% del total)' % (bgp_imp, bgp_pct))
print('Importancia individual features BGP (RF):')
for col in bgp_cols:
    pct = imp_B[col] / total_imp * 100
    print('  %-30s %.4f  (%.2f%%)' % (col, imp_B[col], pct))

print('Top 20 features RF Config B:')
print(imp_B.head(20).to_string())
del X_tr_B, X_va_B, rf_B2; gc.collect()


RF — Importancia BGP total (MDI): 0.0129  (1.29% del total)
Importancia individual features BGP (RF):
  bgp_path_changes               0.0052  (0.52%)
  bgp_reachability               0.0020  (0.20%)
  bgp_updates                    0.0057  (0.57%)
Top 20 features RF Config B:
y_lag1                   0.065236
y_lag2                   0.053038
y_lag3                   0.048055
streak_length            0.036813
delta_p95_loss           0.029339
delta_p95_lat            0.024611
mean_loss                0.022311
lag1_delta_p95_loss      0.020882
lag1_delta_p95_lat       0.020373
lag2_delta_p95_lat       0.019905
lag2_delta_p95_loss      0.019348
p95_loss                 0.015561
lag1_mean_loss           0.014783
lag2_mean_loss           0.014770
std_loss                 0.013703
lag1_p95_loss            0.013565
delta_p95_jitter         0.013101
lag1_delta_p95_jitter    0.011887
lag2_delta_p95_jitter    0.011749
std_lat                  0.011737


48

In [14]:
import os
TMP_DIR = '/srv/work/ablacion_tmp'
X_tr_B = np.load(os.path.join(TMP_DIR, 'X_tr_B.npy'))
X_va_B = np.load(os.path.join(TMP_DIR, 'X_va_B.npy'))
lgb_B2 = lgb.LGBMClassifier(**LGB_PARAMS)
lgb_B2.fit(X_tr_B, y_train)

lgb_imp_B = pd.Series(lgb_B2.feature_importances_, index=FEAT_B).sort_values(ascending=False)
lgb_bgp_imp = lgb_imp_B[bgp_cols].sum()
lgb_total   = lgb_imp_B.sum()
lgb_bgp_pct = lgb_bgp_imp / lgb_total * 100

print('LGB — Importancia BGP total (gain): %.0f  (%.2f%% del total)' % (lgb_bgp_imp, lgb_bgp_pct))
print('Importancia individual features BGP (LGB):')
for col in bgp_cols:
    pct = lgb_imp_B[col] / lgb_total * 100
    print('  %-30s %.0f  (%.2f%%)' % (col, lgb_imp_B[col], pct))

print('Top 20 features LGB Config B:')
print(lgb_imp_B.head(20).to_string())
del X_tr_B, X_va_B, lgb_B2; gc.collect()


LGB — Importancia BGP total (gain): 5450  (4.40% del total)
Importancia individual features BGP (LGB):
  bgp_path_changes               2211  (1.78%)
  bgp_reachability               965  (0.78%)
  bgp_updates                    2274  (1.83%)
Top 20 features LGB Config B:
streak_length             4221
delta_p95_loss            2999
delta_p95_lat             2826
lag1_delta_p95_loss       2490
lag2_delta_p95_loss       2423
bgp_updates               2274
delta_p95_jitter          2230
bgp_path_changes          2211
lag1_delta_p95_jitter     2091
lag2_delta_p95_jitter     2083
hour_sin                  2009
lag1_delta_p95_lat        1968
lag2_delta_p95_lat        1931
lag48_delta_p95_jitter    1863
lag48_delta_p95_loss      1847
delta_mad_jitter          1825
lag48_delta_p95_lat       1810
d_latency                 1737
hour_cos                  1679
delta_mad_lat             1566


0

In [15]:
print('='*60)
print('VEREDICTO H1 — Ablación BGP v2 (Dataset 162 días)')
print('='*60)

drf  = delta(resultados.get('rf_B',  {}), resultados.get('rf_A',  {}), 'qwk')
dlgb = delta(resultados.get('lgb_B', {}), resultados.get('lgb_A', {}), 'qwk')

h1_rf  = drf  <= 0
h1_lgb = dlgb <= 0

rf_status  = '✅' if h1_rf  else '❌'
lgb_status = '✅' if h1_lgb else '❌'

print(f'  RF  ΔB-A QWK = {drf:+.4f}  →  {rf_status}')
print(f'  LGB ΔB-A QWK = {dlgb:+.4f}  →  {lgb_status}')

if h1_rf and h1_lgb:
    print('\n✅ H1 VERIFICADA en ambos modelos')
    print('   Las features BGP NO mejoran el rendimiento predictivo')
    print('   en el dataset ampliado de 162 días (te_features_v6)')
elif h1_rf or h1_lgb:
    print('\n⚠️ H1 PARCIALMENTE VERIFICADA (solo un modelo)')
else:
    print('\n❌ H1 NO VERIFICADA — BGP sí mejora en ambos modelos')

print('\nReferencia v1 (83 días): ΔB-A ≤ 0 en 6/6 pares')
print('='*60)

VEREDICTO H1 — Ablación BGP v2 (Dataset 162 días)
  RF  ΔB-A QWK = -0.0000  →  ✅
  LGB ΔB-A QWK = -0.0113  →  ✅

✅ H1 VERIFICADA en ambos modelos
   Las features BGP NO mejoran el rendimiento predictivo
   en el dataset ampliado de 162 días (te_features_v6)

Referencia v1 (83 días): ΔB-A ≤ 0 en 6/6 pares
